In [1]:
import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import copy
import gymnasium as gym
from tqdm import tqdm
import sys
sys.path.insert(0, "../src/")
from Network import *
from inputFuc import *
from plotting import *
from utilsRL import *
from IPython.display import display, clear_output
import time
import imageio

In [2]:
data_config = dict(default_data_config)
general_config = dict(default_general_config)
train_config = dict(default_train_config)

general_config['save_path'] = "../saved_models/Cartpole/"

#### With P learning ###
dt = general_config["dt"]

# init both teacher and student net
batch = train_config["batch_size"]
general_config["device"] = "cuda" if torch.cuda.is_available() else "cpu"
device = general_config["device"]
print(device)

env = CartPoleCustomize()

n_actions = env.env.action_space.n.item()
# Get the number of state observations
obs, info = env.reset()
n_observations = obs.shape[-1]

actor_config = dict(default_model_config)
critic_config = dict(default_model_config)
actor_config['n_in'] = n_observations
critic_config['n_in'] = n_observations
actor_config['n_out'] = n_actions
critic_config['n_out'] = 1

cpu


# Train from scratch

In [27]:
def episode_interpolate(env, actor, critic, critic_target, optimizer_actor, optimizer_critic):
    # Run one episode
    observation, _ = env.reset()
    actor.reset()
    critic.reset()
    critic_target.reset()

    terminated, truncated = False, False
    total_reward=0
    gamma=0.99

    interpolate = 2
    interp_k = torch.linspace(0, 1, interpolate)
    observations = observation[None,:, :].repeat(interpolate, 1, 1)

    action_step = 0.02

    with torch.no_grad():
        t = 0
        while not terminated and not truncated:
            # Take a random action
            for k in range(1, interpolate):
                u, _ = actor.step(observations[k])
                value,_ = critic.step(observations[k])

            p = torch.softmax(u, dim=1)
            action = np.random.choice(n_actions, p=p[0].cpu().numpy())

            # Step the environment
            observation_next, reward, terminated, truncated,_ = env.step(action)

            observations = observation[None, :, :] + interp_k[:, None, None] * (observation_next - observation)[None, :,  :]

            for k in range(1, interpolate):
                value_next,_ = critic_target.step(observations[k])

            delta = reward + gamma*value_next - value

            actor.prop(F.one_hot(torch.tensor(action), num_classes=n_actions)-p)
            actor.backwardsRL(delta.item(), gamma)
            critic.prop(1.)
            critic.backwardsRL(delta.item(), gamma)

            observation = observation_next
            total_reward+=reward

            t += 1
            if t%50==0:
                optimizer_actor.step()
                optimizer_critic.step()
                optimizer_actor.zero_grad(set_to_none=False)
                optimizer_critic.zero_grad(set_to_none=False)
                


        optimizer_actor.step()
        optimizer_critic.step()
        optimizer_actor.zero_grad(set_to_none=False)
        optimizer_critic.zero_grad(set_to_none=False)
        #soft_update(critic_target, critic, tau=0.5)
        #hard_update(critic_target, critic)
        

    return total_reward, t*action_step

In [84]:
#Change schedular, add temporatue?
actor = buildRLNet(actor_config, general_config).to(device)
critic = buildRLNet(critic_config, general_config).to(device)
critic_target = buildRLNet(critic_config, general_config).to(device)
for p in critic_target.parameters():
    p.requires_grad = False
hard_update(critic_target, critic)

num_episode = 3000

optimizer_actor = torch.optim.Adam(
    actor.parameters(),
    lr=4e-4,
    betas=(0.95, 0.999)
)

optimizer_critic = torch.optim.Adam(
    critic.parameters(),
    lr=5e-3,
    betas=(0.95, 0.999)
)

scheduler_actor = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer_actor,
    T_max = num_episode,
)

scheduler_critic = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer_critic,
    T_max = num_episode,
)

error_record = []
Cum_errors = 0.5

In [85]:
# Create the CartPole environment
R_avg, duration_avg = 0, 0
best_R, best_duration = 0, 0
R_rec = []
for i in range(num_episode):
    total_reward, duration = episode_interpolate(env, actor, critic, critic_target, optimizer_actor, optimizer_critic)
    scheduler_actor.step()
    scheduler_critic.step()
    R_avg = 0.998*R_avg + 0.002*total_reward
    duration_avg = 0.998*duration_avg + 0.002*duration
    hard_update(critic_target, critic)
    if i%100==99:
        R_rec.append(R_avg)
        print(R_avg, duration_avg)
        if R_avg > best_R:
            best_R = R_avg
            best_actor_state = actor.state_dict().copy()
            best_critic_state = critic.state_dict().copy()


env.close() # Close the environment
print("finished")

if best_R > 400:
    torch.save(best_actor_state, general_config['save_path']+"actor%d.ckp"%int(best_R))
    torch.save(best_critic_state, general_config['save_path']+"critic%d.ckp"%int(best_R))

3.885706 0.10208754728144438
12.986676 0.3277648460026644
20.745155 0.5197443730054487
31.568981 0.7946431039583797
46.526352 1.1727046840772386
53.756157 1.3497521779141581
57.51832 1.421852958485963
61.237453 1.4916931269191436
63.757557 1.5417860829803491
78.33779 1.907037786186449
100.900696 2.411963983972742
115.83812 2.7978696791777073
112.238075 2.7304153510262217
101.99088 2.479610919009179
92.84419 2.2537152099256867
84.03576 2.0400305905037244
79.20543 1.9132177515206656
75.45668 1.8163936980198248
72.66247 1.7456593833821978
74.89864 1.7898448886309888
80.29998 1.9314961743549561
101.255516 2.432274671051002
117.43712 2.7611554063148183
129.82025 3.00582146852219
135.9836 3.130332021033167
145.76103 3.323521432424367
152.66289 3.46347554074774
157.77303 3.575949423155062
162.42415 3.668708756213735
165.65746 3.7330836094693023
finished


# Adaptation and visulize

In [82]:
def episode_interpolate_adapt(env, actor, critic, critic_target, 
                              optimizer_actor, optimizer_critic,
                              adapt=False, visualize=False,):
    # Run one episode
    observation, _ = env.reset()
    actor.reset()
    critic.reset()
    critic_target.reset()

    terminated, truncated = False, False
    total_reward=0
    gamma=0.99

    interpolate = 2
    interp_k = torch.linspace(0, 1, interpolate)
    observations = observation[None,:, :].repeat(interpolate, 1, 1)

    action_step = 20 #ms
    if visualize:
        plt.figure(figsize=(5,4))
    frames = []

    with torch.no_grad():
        t = 0
        while not terminated and not truncated:
            # Take a random action
            for k in range(1, interpolate):
                u, _ = actor.step(observations[k])
                value,_ = critic.step(observations[k])

            p = torch.softmax(u, dim=1)
            action = np.random.choice(n_actions, p=p[0].cpu().numpy())

            # Step the environment
            observation_next, reward, terminated, truncated,_ = env.step(action)
            frame = env.render()
            frames.append(frame)
            if visualize:
                plt.imshow(frame)
                plt.axis("off")
                display(plt.gcf())
                clear_output(wait=True)

            observations = observation[None, :, :] + interp_k[:, None, None] * (observation_next - observation)[None, :,  :]

            for k in range(1, interpolate):
                value_next,_ = critic_target.step(observations[k])

            delta = reward + gamma*value_next - value

            if adapt:
                actor.prop(F.one_hot(torch.tensor(action), num_classes=n_actions)-p)
                actor.backwardsRL(delta.item(), gamma)
                critic.prop(1.)
                critic.backwardsRL(delta.item(), gamma)

            observation = observation_next
            total_reward+=reward

            t+=1
            if t%40==0 and adapt:
                optimizer_actor.step()
                optimizer_critic.step()
                optimizer_actor.zero_grad(set_to_none=False)
                optimizer_critic.zero_grad(set_to_none=False)
            if t%200==0 and adapt:
                hard_update(critic_target, critic)

        if adapt:
            optimizer_actor.step()
            optimizer_critic.step()
            optimizer_actor.zero_grad(set_to_none=False)
            optimizer_critic.zero_grad(set_to_none=False)
            

    return total_reward, frames

In [87]:
num_trials = 100
wind = 7.

visualize = True if num_trials<=2 else False
env = WindCartPoleEnv(wind=wind, render_mode="rgb_array")
env = CartPoleCustomize(env=env)  #Do we need time limit?
#env = CartPoleCustomize(env=None, display = True)
actor_state_dict = torch.load(general_config['save_path']+"actor496.ckp", weights_only=True)
critic_state_dict = torch.load(general_config['save_path']+"critic496.ckp", weights_only=True)

actor = buildRLNet(actor_config, general_config).to(device)
critic = buildRLNet(critic_config, general_config).to(device)
critic_target = buildRLNet(critic_config, general_config).to(device)

In [88]:
#Without online adpation
noAdapt_reward = []
actor.load_state_dict(actor_state_dict)
critic.load_state_dict(critic_state_dict)
critic_target.load_state_dict(critic_state_dict)

optimizer_actor = torch.optim.Adam(actor.parameters(), lr=0.)
optimizer_critic = torch.optim.Adam(critic.parameters(), lr=0.,)

for i in range(num_trials):
    total_reward, frames = episode_interpolate_adapt(env, actor, critic, critic_target,
                                                     optimizer_actor, optimizer_critic,
                                                     adapt=False, visualize=visualize,)
    noAdapt_reward.append(total_reward)
    if total_reward <0:
        imageio.mimsave("../Figures/cartpole_noAdapt%.1f.gif"%total_reward, frames, fps=30)

noAdapt_reward = np.array(noAdapt_reward)
print("Average Reward without adaptation: ", noAdapt_reward.mean())

KeyboardInterrupt: 

In [ ]:
#With online adpation
Adapt_reward = []
for i in range(num_trials):
    actor.load_state_dict(actor_state_dict)
    critic.load_state_dict(critic_state_dict)
    critic_target.load_state_dict(critic_state_dict)
    
    optimizer_actor = torch.optim.Adam(
        actor.parameters(),
        lr=1e-3,
        betas=(0.9, 0.9)
    )
    
    optimizer_critic = torch.optim.Adam(
        critic.parameters(),
        lr=0e-3,
        betas=(0.9, 0.999)
    )
    
    total_reward, frames = episode_interpolate_adapt(env, actor, critic, critic_target,
                                                     optimizer_actor, optimizer_critic,
                                                     adapt=True, visualize=visualize,)
    Adapt_reward.append(total_reward)
    if total_reward >700:
        imageio.mimsave("../Figures/cartpole_Adapt%.1f.gif"%total_reward, frames, fps=30)
Adapt_reward = np.array(Adapt_reward)
print("Average Reward with adaptation: ", Adapt_reward.mean())

In [ ]:
#TODO plot the bar with many runs
fig, axs = plt.subplots(1, 2, figsize = (10, 4))

data = [noAdapt_reward, Adapt_reward]
axs[1].violinplot(data, positions=[0,1], showmeans=True)
axs[1].set_xticks([0,1], ['No adapt', 'KP adapt'], fontsize=16)

for i, d in enumerate(data):
    x = np.random.normal(i, 0.03, size=len(d))  # small horizontal jitter
    axs[1].scatter(x, d, alpha=0.3, s=10)

#plt.savefig('./fig/LinRegCf%s.png'%activation)
plt.show()

In [ ]:
#TODO multi-episode learn, faster than BPTT?